In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split
import torch
import torch.nn as nn

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)



In [3]:
train, test = random_split(dataset, lengths=[0.8, 0.2])

train_loader = DataLoader(train, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test, shuffle=True, num_workers=2)

In [4]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [5]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.max_pool(x)
        x = self.relu(self.conv2(x))
        x = self.max_pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        return x
    

model = CNN()
model.forward(images).shape
        

torch.Size([64, 10])

In [6]:
def train(model: CNN, train_loader: DataLoader, val_loader: DataLoader, epochs: int, optimiser: torch.optim.SGD, criterion) -> None:
    for epoch in range(epochs):
        for batch_data, batch_labels in train_loader:
            optimiser.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimiser.step()
        model.eval()
        
        total_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_data, batch_labels in val_loader:
                outputs = model(batch_data)
                loss = criterion(outputs, batch_labels)
                total_loss += loss.item() * batch_data.size(0)
                
                preds = outputs.argmax(dim=1)
                correct += (preds == batch_labels).sum().item()
                total += batch_labels.size(0)
                
        avg_loss = total_loss / total
        accuracy = correct / total
        print(f"Epoch: {epoch} eval loss: {avg_loss:.4f}, accuracy: {accuracy:.4f}")
        model.train()
            
        


optimiser = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
train(
    model=model, 
    train_loader=train_loader, 
    val_loader=test_loader, 
    epochs=10, 
    optimiser=optimiser, 
    criterion=criterion
)

Epoch: 0 eval loss: 0.3087, accuracy: 0.9057
Epoch: 1 eval loss: 0.2022, accuracy: 0.9343
Epoch: 2 eval loss: 0.1533, accuracy: 0.9557
Epoch: 3 eval loss: 0.1174, accuracy: 0.9662
Epoch: 4 eval loss: 0.1053, accuracy: 0.9697
Epoch: 5 eval loss: 0.0932, accuracy: 0.9722
Epoch: 6 eval loss: 0.0873, accuracy: 0.9735
Epoch: 7 eval loss: 0.0849, accuracy: 0.9733
Epoch: 8 eval loss: 0.0827, accuracy: 0.9746
Epoch: 9 eval loss: 0.0757, accuracy: 0.9767
